# Intuitor vs GRPO — MATH Dataset | L4 24GB
### Usando o código real do repositório `sunblaze-ucb/Intuitor`

Baseado em: [Zhao et al., ICLR 2026](https://arxiv.org/abs/2505.19590)  
Repositório: [sunblaze-ucb/Intuitor](https://github.com/sunblaze-ucb/Intuitor)

**Fluxo do notebook:**
1. Verificar GPU → Clonar repo + instalar → Config → Drive → Dataset → Modelo Base
2. *(Opcional)* Avaliar modelo base antes do treino
3. Rewards → LoRA + Config → Treino → Avaliação final

> ⚠️ Selecione **L4** em `Ambiente de execução > Alterar tipo de ambiente`

## Célula 0 — Verificação da GPU

In [ ]:
import subprocess, torch

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print(f"PyTorch  : {torch.__version__}")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU      : {name}")
    print(f"VRAM     : {vram:.1f} GB")
    assert vram >= 20, f"VRAM insuficiente ({vram:.1f} GB). Requer L4 (24 GB)+."
    print("✅ GPU verificada.")

Thu Jun 11 11:35:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   38C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Célula 1 — Clone do repositório e instalação
> Demora ~5–8 min. Rode apenas uma vez por sessão.

In [ ]:
import sys, os

# 1. Clona o repositório oficial
!git clone --depth 1 https://github.com/sunblaze-ucb/Intuitor.git

# 2. Adiciona o src ao Python path — necessário para importar open_r1
REPO_SRC = "/content/Intuitor/open-r1-intuitor/src"
if REPO_SRC not in sys.path:
    sys.path.insert(0, REPO_SRC)
print(f"✅ Path: {REPO_SRC}")

# 3. Dependências core (versões pinadas)
!pip install -q --upgrade pip
!pip install -q \
    transformers==4.51.0 \
    trl==0.17.0 \
    accelerate==1.4.0 \
    peft==0.15.1 \
    datasets==3.2.0

# 4. Dependências específicas do open-r1-intuitor
!pip install -q \
    math-verify \
    latex2sympy2_extended \
    liger-kernel \
    bitsandbytes \
    einops \
    wandb \
    sentencepiece \
    async-lru

# 5. Verifica importação do open_r1
try:
    import open_r1
    print("✅ open_r1 importado com sucesso")
except ImportError as e:
    print(f"❌ Falha: {e}")

print("\n✅ Instalação concluída.")

Cloning into 'Intuitor'...
remote: Enumerating objects: 1034, done.
remote: Counting objects: 100% (1034/1034), done.
remote: Compressing objects: 100% (797/797), done.
remote: Total 1034 (delta 280), reused 765 (delta 198), pack-reused 0 (from 0)
Receiving objects: 100% (1034/1034), 2.18 MiB | 27.52 MiB/s, done.
Resolving deltas: 100% (280/280), done.
✅ Path: /content/Intuitor/open-r1-intuitor/src
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 84.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.
✅ open_r1 importado com sucesso

✅ Instalação concluída.


## Célula 2 — Imports e config global

Troque `MODE` entre `"intuitor"` e `"grpo"` para comparar.

**Correções aplicadas:**
- `TEMPERATURE = 1.1` → força diversidade entre as G gerações (resolve `loss=0`)
- `SYSTEM_PROMPT` estruturado com `<think>/<answer>` → modelo base entende o formato esperado
- Warnings do `latex2sympy2_extended` silenciados

In [ ]:
import re, os, math, warnings, logging
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed, TrainerCallback
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig, TaskType

# Silencia warnings do latex2sympy2_extended
logging.getLogger("latex2sympy2_extended.math_normalization").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning)

# ── CONFIG ────────────────────────────────────────────────────────────────────
MODEL_NAME      = "Qwen/Qwen2.5-1.5B"
N_TRAIN         = 1000
SEED            = 42
LEARNING_RATE   = 3e-6
LORA_R          = 16
LORA_ALPHA      = 32
NUM_EPOCHS      = 1
NUM_GENERATIONS = 8
BETA            = 0.007
TEMPERATURE     = 0.9

# ÚNICO SYSTEM PROMPT PARA AMBOS OS MODOS (Garante controle científico das variáveis)
SYSTEM_PROMPT = (
    "You are a helpful AI Assistant, designed to provided well-reasoned "
    "and detailed responses. You FIRST think about the reasoning process "
    "step by step and then provide the user with the answer. "
    "Please enclose your final answer in the box: \\boxed{Your Answer}. "
    "Please stop generation immediately after outputing the box."
)

# ── MODO ─────────────────────────────────────────────────────────────────────
# "intuitor" → INTUITORTrainer (self-certainty interna, sem labels)
# "grpo"     → GRPOTrainer    (accuracy + format reward, usa labels)
MODE = "grpo"   # ← altere aqui

OUTPUT_DIR = f"qwen-1.5b-{MODE}-gsm8k1000"
print(f"Modo        : {MODE.upper()}")
print(f"Modelo      : {MODEL_NAME}")
print(f"Output      : {OUTPUT_DIR}")
set_seed(SEED)

Modo        : GRPO
Modelo      : Qwen/Qwen2.5-1.5B
Output      : qwen-1.5b-grpo-gsm8k1000


## Célula 3 — Google Drive
Salva checkpoints para não perder em desconexões.

In [ ]:
USE_DRIVE = True  # ← False se não quiser montar

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = f"/content/drive/MyDrive/intuitor-ic/{OUTPUT_DIR}"
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f"✅ Checkpoints em: {SAVE_DIR}")
else:
    SAVE_DIR = f"/content/{OUTPUT_DIR}"
    print(f"⚠️  Salvando localmente (perdido ao desconectar): {SAVE_DIR}")

Mounted at /content/drive
✅ Checkpoints em: /content/drive/MyDrive/intuitor-ic/qwen-1.5b-grpo-gsm8k1000


## Célula 4 — Dataset MATH (500 problemas)

In [ ]:
# Célula 4 — Dataset GSM8K
def make_conversation(example):
    """
    Replica make_conversation() do repositório.
    Gabarito formatado como \\boxed{número} para o math_verify parsear corretamente.

    Por que \\boxed{}?
    - parse("42")          → lista vazia → accuracy_reward = None → loss = 0  ❌
    - parse("\\boxed{42}") → expressão válida → reward funciona              ✅
    """
    prompt = []
    if SYSTEM_PROMPT:
        prompt.append({"role": "system", "content": SYSTEM_PROMPT})
    prompt.append({"role": "user", "content": example["question"]})

    # Extrai o número e formata como LaTeX para o math_verify
    gold_number = example["answer"].split("####")[-1].strip()
    gold_answer = f"\\boxed{{{gold_number}}}"

    return {"prompt": prompt, "solution": gold_answer}

print("Carregando GSM8K...")
train_full = load_dataset("openai/gsm8k", "main", split="train")
train_full = train_full.shuffle(seed=SEED)
train_raw  = train_full.select(range(N_TRAIN))
train_dataset = train_raw.map(make_conversation, batched=False)
train_dataset = train_dataset.remove_columns(
    [c for c in train_dataset.column_names if c not in ["prompt", "solution"]]
)

test_raw = (
    load_dataset("openai/gsm8k", "main", split="test")
    .shuffle(seed=SEED)
    .select(range(200))
)

print(f"\n✅ Treino : {len(train_dataset)} problemas")
print(f"✅ Teste  : {len(test_raw)} problemas")
print("\nExemplo de prompt:")
print(train_dataset[0]["prompt"])
print("\nExemplo de solution (LaTeX para math_verify):")
print(train_dataset[0]["solution"])  # deve mostrar \boxed{42} ou similar

Carregando GSM8K...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


✅ Treino : 1000 problemas
✅ Teste  : 200 problemas

Exemplo de prompt:
[{'content': 'You are a helpful AI Assistant, designed to provided well-reasoned and detailed responses. You FIRST think about the reasoning process step by step and then provide the user with the answer. Please enclose your final answer in the box: \\boxed{Your Answer}. Please stop generation immediately after outputing the box.', 'role': 'system'}, {'content': 'Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?', 'role': 'user'}]

Exemplo de solution (LaTeX para math_verify):
\boxed{16}


## Célula 5 — Modelo e Tokenizer

In [ ]:
print(f"Carregando tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = "left"    # padrão para modelos base
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Carregando modelo (bf16 + flash-attn 2)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)
model.config.pad_token_id = tokenizer.pad_token_id

# NOTA: NÃO chamar enable_input_require_grads() aqui.
# O PEFT só injeta as matrizes LoRA quando o Trainer é instanciado com peft_config.
# Chamar antes quebra o grafo do PyTorch. A ativação correta é via callback (Célula 8).

params  = sum(p.numel() for p in model.parameters()) / 1e9
vram_gb = torch.cuda.memory_allocated() / 1e9
print(f"\n✅ Modelo carregado")
print(f"   Parâmetros : {params:.2f} B")
print(f"   VRAM usada : {vram_gb:.2f} GB de 24 GB")

Carregando tokenizer: Qwen/Qwen2.5-1.5B


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]


✅ Modelo carregado
   Parâmetros : 1.54 B
   VRAM usada : 3.09 GB de 24 GB


## Célula 6 — *(Opcional)* Avaliação do modelo BASE

Roda **antes do treino** para registrar o baseline.
Resultado guardado em `base_score` para comparação final.

In [ ]:
from math_verify import LatexExtractionConfig, parse, verify
from latex2sympy2_extended import NormalizationConfig

def evaluate_math(model, tokenizer, dataset, desc="", restore_train=False):
    """
    Avaliação pass@1 greedy com math_verify para GSM8K.

    O dataset de teste vem direto do load_dataset (sem map), portanto:
      - campo do problema : example['question']
      - campo do gabarito : example['answer']  (contém '#### número')

    O gabarito é extraído e formatado como \\boxed{número} aqui,
    igual ao que a Célula 4 faz para o dataset de treino.
    """
    model.eval()
    passed = 0
    total  = len(dataset)
    print(f"\n{'='*60}")
    print(f"AVALIAÇÃO GSM8K TEST ({total} exemplos) {desc}")
    print(f"{'='*60}")

    for i, example in enumerate(dataset):
        messages = []
        if SYSTEM_PROMPT:
            messages.append({"role": "system", "content": SYSTEM_PROMPT})
        messages.append({"role": "user", "content": example["question"]})

        try:
            prompt_str = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        except Exception:
            prompt_str = SYSTEM_PROMPT + "\n" + example["question"] + "\n"

        inputs = tokenizer(
            prompt_str, return_tensors="pt", truncation=True, max_length=768
        ).to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        completion = tokenizer.decode(
            out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
        )

        # Extrai gabarito do campo 'answer' original do GSM8K
        # mesmo processo da Célula 4: corta no '####' e formata como \boxed{}
        gold_number = example["answer"].split("####")[-1].strip()
        gold_str    = f"\\boxed{{{gold_number}}}"

        gold_parsed = parse(gold_str, extraction_mode="first_match")
        if len(gold_parsed) > 0:
            answer_parsed = parse(
                completion,
                extraction_config=[
                    LatexExtractionConfig(
                        normalization_config=NormalizationConfig(
                            nits=False, malformed_operators=False,
                            basic_latex=True, equations=True, boxed="all", units=True,
                        ),
                        boxed_match_priority=0,
                        try_extract_without_anchor=False,
                    )
                ],
                extraction_mode="first_match",
            )
            try:
                if verify(gold_parsed, answer_parsed):
                    passed += 1
            except Exception:
                pass

        if (i + 1) % 50 == 0:
            print(f"  [{i+1}/{total}] pass@1 parcial: {passed/(i+1):.4f}")

    score = passed / total
    print(f"\n{'='*60}")
    print(f"RESULTADO — pass@1 : {score:.4f} ({score*100:.2f}%)")
    print(f"Corretos           : {passed}/{total}")
    print(f"{'='*60}")

    if restore_train:
        model.train()

    return score

print("Iniciando avaliação do modelo BASE...")
base_score = evaluate_math(
    model, tokenizer, test_raw,
    desc="— BASE MODEL (0 épocas)",
    restore_train=False,
)
print(f"\n✅ BASE SCORE registrado: {base_score*100:.2f}%")
print("Prossiga para as células de treino.")

Iniciando avaliação do modelo BASE...

AVALIAÇÃO GSM8K TEST (200 exemplos) — BASE MODEL (0 épocas)


KeyboardInterrupt: 

## Célula 7 — Reward Functions

Importadas diretamente do repositório clonado.

**Por que manter o `format_reward` para GRPO:**
- Remover o `format_reward` não resolve o `loss=0` — sem o formato estruturado,
  o modelo base também erra o `accuracy_reward` → advantage continua zero
- A solução correta é `TEMPERATURE=1.1` (Célula 2), que garante diversidade
  entre as G gerações → variância no advantage → gradiente real
- O `format_reward` combinado com o novo `SYSTEM_PROMPT` cria um sinal gradual:
  gerações que acertam o formato ganham reward maior → aprendizado do formato first

In [ ]:
from open_r1.rewards import accuracy_reward, format_reward

if MODE == "intuitor":
    # INTUITORTrainer computa self-certainty internamente — sem rewards externos
    reward_funcs = []
    print("✅ Modo INTUITOR")
    print("   Reward : self-certainty interna (INTUITORTrainer)")
    print("   reward_funcs=[] — nenhum reward externo necessário")
else:
    reward_funcs = [accuracy_reward, format_reward]
    print("✅ Modo GRPO")
    print("   Rewards: accuracy_reward + format_reward (open_r1.rewards)")
    print("   NOTA: format_reward requer <think>...</think><answer>...</answer>")
    print("         SYSTEM_PROMPT estruturado garante que o modelo tente o formato")

✅ Modo GRPO
   Rewards: accuracy_reward + format_reward (open_r1.rewards)
   NOTA: format_reward requer <think>...</think><answer>...</answer>
         SYSTEM_PROMPT estruturado garante que o modelo tente o formato


/content/Intuitor/open-r1-intuitor/src/open_r1/utils/ioi/morph_client.py:497: SyntaxWarning: invalid escape sequence '\/'
  find . -type f -o -type d | sed -e 's/[^-][^\/]*\//  |/g' -e 's/|\([^ ]\)/|-\1/' >&2
/content/Intuitor/open-r1-intuitor/src/open_r1/utils/ioi/piston_client.py:43: SyntaxWarning: invalid escape sequence '\ '
  sed -i '/app.use(body_parser.urlencoded/c\    app.use(body_parser.urlencoded({ extended: true, limit: \"512mb\" }));' src/index.js


## Célula 8 — LoRA + GRPOConfig

In [ ]:
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "up_proj", "down_proj", "gate_proj"],
    task_type=TaskType.CAUSAL_LM,
    lora_dropout=0.05,
    bias="none",
)

training_args = GRPOConfig(
    output_dir=SAVE_DIR,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,

    # ── Memória (L4 24 GB) ──────────────────────────────────────────────────
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    bf16=True,
    fp16=False,

    # ── Hiperparâmetros do paper ─────────────────────────────────────────────
    num_generations=NUM_GENERATIONS,
    max_prompt_length=512,
    max_completion_length=768,
    temperature=TEMPERATURE,           # 1.1 — forçar diversidade
    beta=BETA,

    # ── Logging ─────────────────────────────────────────────────────────────
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,

    # ── Misc ────────────────────────────────────────────────────────────────
    use_vllm=False,
    report_to="none",
    seed=SEED,
    #
    epsilon=0.2,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
)

steps = N_TRAIN // training_args.per_device_train_batch_size * NUM_EPOCHS
print(f"✅ Config definida")
print(f"   Batch efetivo  : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Steps estimados: ~{steps}")
print(f"   Tempo estimado : ~1–2h na L4")

✅ Config definida
   Batch efetivo  : 8
   Steps estimados: ~1000
   Tempo estimado : ~1–2h na L4


## Célula 9 — Treino

**Correção do warning `Gradients will be None`:**
O warning ocorre porque `gradient_checkpointing` + PEFT exige que
`enable_input_require_grads()` seja chamado **no modelo PEFT já montado**,
não no modelo base. A forma correta é via `TrainerCallback.on_train_begin`,
que é chamado após o Trainer injetar as matrizes LoRA.

In [ ]:
from open_r1.intuitor_trainer import INTUITORTrainer

# CORREÇÃO: enable_input_require_grads() via callback — após o PEFT montar o modelo
# Chamar antes quebra o grafo do PyTorch; dentro do callback é seguro
class EnableGradsCallback(TrainerCallback):
    """
    Ativa require_grads nos embeddings após o Trainer injetar o LoRA.
    Necessário para gradient_checkpointing funcionar com PEFT.
    Sem isso: 'None of the inputs have requires_grad=True. Gradients will be None'
    """
    def on_train_begin(self, args, state, control, model=None, **kwargs):
        model.enable_input_require_grads()
        print("✅ enable_input_require_grads() ativado (pós-PEFT)")

print("=" * 60)
print(f"TREINO {MODE.upper()} | {MODEL_NAME}")
print(f"Dataset: MATH train ({N_TRAIN} problemas)")
print(f"Config : épocas={NUM_EPOCHS} | G={NUM_GENERATIONS} | T={TEMPERATURE} | β={BETA} | lr={LEARNING_RATE} | LoRA r={LORA_R}")
print("=" * 60)

TrainerClass = INTUITORTrainer if MODE == "intuitor" else GRPOTrainer
print(f"Trainer: {'INTUITORTrainer (self-certainty interna)' if MODE == 'intuitor' else 'GRPOTrainer (reward externo)'}")

trainer = TrainerClass(
    model=model,
    reward_funcs=reward_funcs,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    callbacks=[EnableGradsCallback()],
)

trainer.train()

print("\n✅ Treino concluído!")
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Modelo salvo em: {SAVE_DIR}")

TREINO GRPO | Qwen/Qwen2.5-1.5B
Dataset: MATH train (1000 problemas)
Config : épocas=1 | G=8 | T=0.9 | β=0.007 | lr=3e-06 | LoRA r=16
Trainer: GRPOTrainer (reward externo)


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


✅ enable_input_require_grads() ativado (pós-PEFT)


Step,Training Loss
5,0.000000
10,0.000000
15,0.000000
20,0.000000
25,0.000000
30,0.000000
35,0.000000
40,0.000000
45,0.000000
50,0.000000



✅ Treino concluído!
Modelo salvo em: /content/drive/MyDrive/intuitor-ic/qwen-1.5b-grpo-gsm8k1000


In [ ]:
from peft import PeftModel

# Carrega modelo base
model_eval = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# Carrega checkpoint-700
model_eval = PeftModel.from_pretrained(
    model_eval,
    f"{SAVE_DIR}/checkpoint-1000",  # ← mudou de 250 para 700
)

tokenizer_eval = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer_eval.padding_side = "left"
if tokenizer_eval.pad_token is None:
    tokenizer_eval.pad_token = tokenizer_eval.eos_token

# Avalia
score_1000 = evaluate_math(
    model_eval, tokenizer_eval, test_raw,
    desc="— checkpoint-1000",
    restore_train=False,
)

BASE_SCORE = 0.0150  # fixo — medido anteriormente
print(f"\n✅ Score checkpoint-1000: {score_1000*100:.2f}%")
print(f"   Delta vs base: {(score_1000 - BASE_SCORE)*100:+.2f} p.p.")


AVALIAÇÃO GSM8K TEST (200 exemplos) — checkpoint-1000
  [50/200] pass@1 parcial: 0.6800
  [100/200] pass@1 parcial: 0.6800
  [150/200] pass@1 parcial: 0.7067
  [200/200] pass@1 parcial: 0.7100

RESULTADO — pass@1 : 0.7100 (71.00%)
Corretos           : 142/200

✅ Score checkpoint-1000: 71.00%
   Delta vs base: +69.50 p.p.


## Célula 10 — Avaliação final e comparação

In [ ]:
final_score = evaluate_math(
    model, tokenizer, test_raw,
    desc=f"— {MODE.upper()} (pós-treino)",
    restore_train=True,   # pós-treino: restaura model.train()
)

# Baselines do paper — Tabela 1, Qwen2.5-1.5B, MATH-500
print("\nComparação completa:")
print(f"  BASE (0 épocas)             : {base_score:.4f} ({base_score*100:.2f}%)")

baselines = {
    "Intuitor 1.5B base / 1 época (paper)": 0.412,
    "GRPO    1.5B base / 1 época (paper)" : 0.398,
}
for name, b in baselines.items():
    print(f"  {name}: {b:.3f}")

print(f"\n  Seu {MODE.upper()} (pós-treino)  : {final_score:.4f} ({final_score*100:.2f}%)")
print(f"  Delta vs base               : {(final_score - base_score)*100:+.2f} p.p.")

In [ ]:
from peft import PeftModel
base_score = 0.015
print("\nComparação completa:")
print(f"  BASE Instruct + GSM8K (0 épocas) : {base_score:.4f} ({base_score*100:.2f}%)")
final_score = evaluate_math(
    model, tokenizer, test_raw,
    desc=f"— {MODE.upper()} (pós-treino)",
    restore_train=True,
)
print(f"  {MODE.upper()} pós-treino             : {final_score:.4f} ({final_score*100:.2f}%)")
print(f"  Delta vs base                    : {(final_score - base_score)*100:+.2f} p.p.")
print()
print("  NOTA: baselines do paper (Tabela 1) são para MATH-500 com modelo Base.")
print("        Não são comparáveis diretamente com GSM8K + modelo Instruct.")
print("        Comparação válida para a IC: Intuitor vs GRPO nas mesmas condições.")
print()
print("  Para fechar a tabela da IC, rode novamente com MODE='grpo' e registre:")
print(f"  GRPO pós-treino → anote o pass@1 e o delta vs base")


Comparação completa:
  BASE Instruct + GSM8K (0 épocas) : 0.0150 (1.50%)

AVALIAÇÃO GSM8K TEST (200 exemplos) — GRPO (pós-treino)
  [50/200] pass@1 parcial: 0.0000


KeyboardInterrupt: 

---
## Diferenças declaráveis para a IC

| Aspecto | Paper / Repo | Este notebook | Status |
|---|---|---|---|
| Modelo | `Qwen2.5-1.5B` base | `Qwen2.5-1.5B` base | ✅ alinhado |
| `INTUITORTrainer` | repo clonado | repo clonado | ✅ alinhado |
| `accuracy_reward` | `open_r1.rewards` | `open_r1.rewards` | ✅ alinhado |
| Formato do prompt | lista de dicts | lista de dicts | ✅ alinhado |
| Temperatura | 0.9 | 1.1 | ⚠️ ajuste necessário para L4 single-GPU |
| Hardware | multi-GPU + ZeRO | L4 single-GPU | ⚠️ declarar |
| Fine-tuning | full fine-tuning | LoRA r=16 | ⚠️ declarar |
| N_TRAIN | 7.500 | 750 | ⚠️ declarar |
| dataset | MATH | GSM8K | ⚠️ declarar |

**Texto sugerido para a IC:**
> *"Por limitações de hardware (L4 24 GB, single-GPU), utilizamos LoRA r=16 em vez de full
> fine-tuning e reduzimos o dataset de treino para 500 exemplos. A temperatura de amostragem
> foi ajustada para 1.1 (vs 0.9 do paper) para garantir diversidade suficiente entre as G
> gerações em ambiente single-GPU. O `INTUITORTrainer` e as reward functions foram utilizados
> diretamente do repositório oficial."*

**Referência:**
```
@article{zhao2025learning,
  title={Learning to Reason without External Rewards},
  author={Zhao, Xuandong and Kang, Zhewei and Feng, Aosong and Levine, Sergey and Song, Dawn},
  journal={arXiv preprint arXiv:2505.19590},
  year={2025}
}
```

| mmodel| dataset | lr | beta | temp | traning-steps| pass@1 | base|
|---|---|---|---|---|---|---|---|
|qwen-1.5b-instruct-grpo|gsm8k|2e-5|0.005|1.1|750|78.5%|68.5|
|qwen-1.5b-instruct-intuitor|gsm8k|2e-5|0.005|1.1|750|58.5%|68.5|